# Voice Cloning with Qwen3-TTS (Colab)

Clone a voice from a short reference clip and generate new speech in that voice, using the `Qwen3-TTS-12Hz-1.7B-Base` model.

**Before you start:** In Colab, go to `Runtime > Change runtime type` and select a **GPU** (T4 is fine).

## 1. Install dependencies

This pins compatible versions so pip doesn't spend ages resolving conflicts. We skip FlashAttention 2 by default — it's a slow, fragile source build on Colab and isn't required; the model runs fine (just a bit slower) with the default attention implementation. There's an optional cell below if you want to try installing it anyway.

In [ ]:
%%capture
!pip install -q -U "qwen-tts==0.1.1" "gradio==5.50.0" "huggingface_hub>=0.34,<1.0" "transformers==4.57.3" "accelerate==1.12.0"


Restart note: if this is the *first* install in a fresh Colab runtime, you normally don't need to restart. If you previously had different package versions loaded in this session, go to `Runtime > Restart session` once, then re-run from here.

In [ ]:
import importlib
for pkg in ["qwen_tts", "gradio", "transformers", "accelerate"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg}: {getattr(mod, '__version__', 'unknown')}")


### (Optional) FlashAttention 2

Only run this if you specifically want it. It compiles from source and can take 10+ minutes on Colab, and sometimes fails depending on the assigned GPU/CUDA combo. Skip it and go straight to Section 2 if you just want things working.

In [ ]:
# Optional — skip unless you need it
# !pip install -q -U flash-attn --no-build-isolation


## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — go to Runtime > Change runtime type and select a GPU, then re-run.")


## 3. Load the voice-clone model

This downloads the ~4GB `Qwen3-TTS-12Hz-1.7B-Base` weights from Hugging Face the first time it runs, so it can take a few minutes.

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map=DEVICE,
    dtype=DTYPE,
    # attn_implementation="flash_attention_2",  # uncomment if you installed flash-attn above
)
print("Model loaded on", DEVICE)


## 4. Upload your reference audio

Upload a clean 3–10 second WAV/MP3 clip of the voice you want to clone (one speaker, minimal background noise).

In [ ]:
from google.colab import files

uploaded = files.upload()
ref_audio_path = next(iter(uploaded))
print("Using reference audio:", ref_audio_path)


## 5. Set the reference transcript

Type exactly what is said in the reference clip. This greatly improves cloning quality. If you don't know/don't want to provide it, you can instead use `x_vector_only_mode=True` further down (lower quality, no transcript needed).

In [ ]:
ref_text = "Type the exact transcript of your reference audio here."


## 6. Generate cloned speech

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

text_to_speak = "Hello! This is a test of voice cloning with Qwen3 TTS."
language = "English"  # or "Auto" to let the model detect it

wavs, sr = model.generate_voice_clone(
    text=text_to_speak,
    language=language,
    ref_audio=ref_audio_path,
    ref_text=ref_text,
    # x_vector_only_mode=True,  # uncomment to skip needing ref_text (lower quality)
)

output_path = "output_voice_clone.wav"
sf.write(output_path, wavs[0], sr)
print("Saved:", output_path)
display(Audio(output_path))


## 7. (Optional) Generate multiple sentences without re-processing the reference

Build the voice-clone prompt once with `create_voice_clone_prompt`, then reuse it across as many `generate_voice_clone` calls as you like — this skips recomputing reference features each time.

In [ ]:
voice_clone_prompt = model.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=ref_text,
)

sentences = [
    "This is the first cloned sentence.",
    "And here is a second one, in the same voice.",
]

wavs, sr = model.generate_voice_clone(
    text=sentences,
    language=["English", "English"],
    voice_clone_prompt=voice_clone_prompt,
)

for i, w in enumerate(wavs):
    path = f"output_voice_clone_{i}.wav"
    sf.write(path, w, sr)
    print("Saved:", path)
    display(Audio(path))


## 8. (Optional) Simple Gradio UI

A minimal web UI for uploading a reference clip, typing text, and getting cloned audio back — handy if you want to try several sentences quickly.

In [ ]:
import gradio as gr

def clone_voice(ref_audio, ref_transcript, text, language):
    if ref_audio is None or not text.strip():
        return None
    wavs, sr = model.generate_voice_clone(
        text=text,
        language=language or "Auto",
        ref_audio=ref_audio,
        ref_text=ref_transcript or None,
        x_vector_only_mode=not bool(ref_transcript),
    )
    out_path = "gradio_output.wav"
    sf.write(out_path, wavs[0], sr)
    return out_path

demo = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Audio(sources=["upload", "microphone"], type="filepath", label="Reference audio (3-10s)"),
        gr.Textbox(label="Reference transcript (optional but recommended)"),
        gr.Textbox(label="Text to speak", lines=3),
        gr.Textbox(label="Language (e.g. English, Chinese, Auto)", value="Auto"),
    ],
    outputs=gr.Audio(label="Cloned speech"),
    title="Qwen3-TTS Voice Cloning",
)

demo.launch(share=True, debug=False)
